# Icechunk Storage for SPI and SPEI

This notebook shows the Icechunk interface end to end on a small synthetic dataset:
write a prepared input repository, compute SPI-3 and SPEI-3 from a committed snapshot,
commit the results as a new snapshot, and verify the committed values against the
NumPy API.

Icechunk is an optional extra. It needs Python 3.11 or later and Zarr v3:

```bash
uv sync --extra icechunk        # in this repository
pip install 'climate_indices[icechunk]'   # in your own environment
```

Nothing here reaches the network, and the whole notebook runs in a few seconds.

In [ ]:
from pathlib import Path
import shutil
import tempfile

from dask.callbacks import Callback
import icechunk
import numpy as np
import pandas as pd
import xarray as xr

from climate_indices import compute, spei, spi
from climate_indices.icechunk_io import open_icechunk, write_icechunk
from climate_indices.indices import Distribution

work_dir = Path(tempfile.mkdtemp(prefix="icechunk-example-"))
work_dir

## 1. Prepare an input repository

A prepared input is a Dataset whose variables carry the observations: precipitation for
SPI, precipitation and PET for SPEI. The variable names below are this example's
convention, not a library requirement — you select the variables you pass to the index
functions.

`write_icechunk` writes the Dataset in one session and commits it. Its default mode
`w-` refuses a group that already holds data, so a prepared input cannot be overwritten
by accident. The helper returns the committed snapshot ID.

In [ ]:
rng = np.random.default_rng(7)
precip = rng.gamma(2.0, 30.0, (360, 2, 2))
precip[0, 0, 0] = np.nan     # one missing value
precip[12, 1, 1] = 0.0       # one exact zero
pet = rng.gamma(2.0, 2.0, (360, 2, 2))
time = pd.date_range("1981-01-01", periods=360, freq="MS")  # 30 years of months

inputs = xr.Dataset(
    {
        "precip": (("time", "lat", "lon"), precip, {"units": "mm"}),
        "pet": (("time", "lat", "lon"), pet, {"units": "mm"}),
    },
    coords={"time": time, "lat": [40.0, 41.0], "lon": [0.0, 1.0]},
)

input_repo = icechunk.Repository.create(
    icechunk.local_filesystem_storage(str(work_dir / "prepared_input.icechunk"))
)
input_snapshot = write_icechunk(inputs, input_repo, message="Prepared synthetic input")
assert input_snapshot == input_repo.lookup_branch("main")
input_snapshot

## 2. Read a committed snapshot

`open_icechunk` opens the branch's committed snapshot. The read-only session fixes the
snapshot at open time, so a later commit by another writer cannot change what you read,
and the reader keeps the stored chunks instead of loading them.

Put the full time axis in one chunk before fitting: the index functions scale and fit
along time, and one time chunk keeps that work in a single block.
Keep the reader open until all lazy calculations and writes finish.
The final cleanup cell closes this reader and removes the temporary repositories.
If execution fails or you interrupt the kernel, run that cell before you retry.

In [ ]:
opened = open_icechunk(input_repo)
print(dict(opened.sizes), "| stored chunks:", opened["precip"].chunks)
precip_series = opened["precip"].chunk({"time": -1})
pet_series = opened["pet"].chunk({"time": -1})
assert precip_series.chunks[0] == (360,), "the whole time axis must be one chunk"
assert pet_series.chunks[0] == (360,)

## 3. Compute SPI and SPEI

Pass the labeled `DataArray` to the index function. The xarray path infers the data
start year from the time coordinate, and it keeps the calculation lazy until something
consumes it. `data_start_year` is the argument the plain NumPy path needs explicitly.

SPI and SPEI require `calibration_year_initial` and `calibration_year_final`.
They reject the older `calibration_start_year` and `calibration_end_year` keywords.

In [ ]:
executed_tasks = []
with Callback(pretask=lambda key, dsk, state: executed_tasks.append(key)):
    spi_lazy = spi(
        precip_series,
        scale=3,
        distribution=Distribution.gamma,
        periodicity=compute.Periodicity.monthly,
        calibration_year_initial=1981,
        calibration_year_final=2010,
    )
    spei_lazy = spei(
        precip_series,
        pet_series,
        scale=3,
        distribution=Distribution.gamma,
        periodicity=compute.Periodicity.monthly,
        calibration_year_initial=1981,
        calibration_year_final=2010,
    )
assert not executed_tasks, "index construction must not execute Dask tasks"
assert spi_lazy.chunks is not None, "an Icechunk read is Dask-backed and still lazy"
spi_lazy

## 4. Commit the results

`write_icechunk` opens one writable session, calls `icechunk.xarray.to_icechunk`, and
commits. Use `to_icechunk` for Icechunk, never `Dataset.to_zarr`: it records the chunk
writes of Dask workers in the session, which is what makes an eager or distributed write
commit correctly.

A write that fails before or during the commit leaves the committed snapshot in place.
A second write with the default `w-` mode is rejected rather than replacing the group.

In [ ]:
results = xr.Dataset({"spi": spi_lazy, "spei": spei_lazy})
output_repo = icechunk.Repository.open_or_create(
    icechunk.local_filesystem_storage(str(work_dir / "indices.icechunk"))
)
index_snapshot = write_icechunk(results, output_repo, message="Compute SPI-3 and SPEI-3")
assert index_snapshot == output_repo.lookup_branch("main")
print("committed snapshot:", index_snapshot)

try:
    write_icechunk(results, output_repo, message="Must be rejected")
except FileExistsError as error:
    print("default mode refuses to replace the group:", error)
else:
    raise AssertionError("the default w- mode must reject an existing group")

## 5. Verify the committed values

Reopen the branch and compare the committed indices against the plain NumPy path, which
takes the raw arrays and an explicit `data_start_year`. The first two time steps are
missing because the 3-month window has no complete value there.

In [ ]:
oracle_kwargs = {
    "scale": 3,
    "distribution": Distribution.gamma,
    "data_start_year": 1981,
    "periodicity": compute.Periodicity.monthly,
    "calibration_year_initial": 1981,
    "calibration_year_final": 2010,
}
expected_spi = spi(precip, **oracle_kwargs)
expected_spei = spei(precip, pet, **oracle_kwargs)

with open_icechunk(output_repo) as committed:
    committed_spi = committed["spi"].compute()
    committed_spei = committed["spei"].compute()
    np.testing.assert_allclose(committed_spi.values, expected_spi, rtol=1e-6, atol=1e-8, equal_nan=True)
    np.testing.assert_allclose(committed_spei.values, expected_spei, rtol=1e-6, atol=1e-8, equal_nan=True)
    assert committed_spi.time.identical(inputs.time)
    assert np.isnan(committed_spi.values[:2]).all(), "the first windowed steps are missing"
    print("committed SPI and SPEI match the NumPy path")

## 6. Snapshots are immutable history

Every commit adds a snapshot and moves the branch. The snapshot ID stays a durable
handle: read an older one with `readonly_session(snapshot_id=...)`, and name one for
later use with a tag. `mode="w"` replaces the group in a new snapshot; it does not
rewrite the old one.

In [ ]:
replacement = xr.Dataset({"spi": spi_lazy + 1.0, "spei": spei_lazy})
replacement_snapshot = write_icechunk(
    replacement, output_repo, message="Recompute with a shifted SPI", mode="w"
)
assert replacement_snapshot != index_snapshot

output_repo.create_tag("spi-3-shifted", snapshot_id=replacement_snapshot)
assert output_repo.lookup_tag("spi-3-shifted") == replacement_snapshot

with xr.open_zarr(output_repo.readonly_session(snapshot_id=replacement_snapshot).store, consolidated=False) as replaced:
    xr.testing.assert_allclose(replaced.compute(), replacement.compute())

with xr.open_zarr(output_repo.readonly_session(snapshot_id=index_snapshot).store, consolidated=False) as older:
    older_spi = older["spi"].compute()
    assert not np.allclose(older_spi.values, replacement["spi"].compute().values, equal_nan=True)
    print("branch head:", replacement_snapshot, "| tagged:", output_repo.list_tags())

## Notes for real workloads

- **Distributed writes.** With a Dask client, keep the client open until `write_icechunk`
  returns. `to_icechunk` tracks the worker writes in the session and commits once they are
  complete; `Dataset.to_zarr` does not.
- **One writer per branch.** Icechunk detects a stale session and rejects its commit.
  Run one writer per branch, or write to separate branches and merge through the data.
- **Storage.** `local_filesystem_storage` is not safe for concurrent commits from several
  processes. Use an S3-compatible object store when more than one writer commits at once.
- **Reading is snapshot-stable.** An open reader keeps its original snapshot.
  After you finish its lazy calculations, close it and reopen the repository branch
  to read the latest committed snapshot.

Further reading:

- [Write outputs](https://climate-indices.readthedocs.io/en/stable/writing-outputs.html#read-and-write-icechunk-snapshots) —
  the same interface in the user guide.
- `tests/test_icechunk_io.py` — round-trip, parity, and session-semantics tests.
- `notebooks/zarr_dask_spi_spei.ipynb` — the full chunking and Dask scheduling story.

In [ ]:
if "opened" in globals():
    opened.close()
shutil.rmtree(work_dir, ignore_errors=True)
print("removed", work_dir)